# **Оценивание качества моделей выявления аномалий в кибербезопасности**

**Продолжительность:** 80 минут. **Среда:** Jupyter Notebook или Google Colab.

## Учебный датасет и протокол эксперимента

Файл `anomaly_quality_practice.csv` содержит новый синтетический набор, который отличается от данных лекции: три хоста NODE-11, NODE-12 и NODE-13, другие даты, фон и инциденты. Каждая минута каждого периода представлена для всех трёх хостов без пропусков.

| Часть | Период, правая граница не включается | Назначение |
|---|---|---|
| calibration | 03.02.2026–04.02.2026 | Подтверждённая норма для порогов |
| normal_audit | 04.02.2026–05.02.2026 | Новая подтверждённая норма |
| monitoring | 05.02.2026–07.02.2026 | Двухсуточный поток; сначала без использования истины |
| shifted_normal | 08.02.2026–09.02.2026 | Подтверждённая норма с возросшей нагрузкой |

| Поле | Содержание и использование |
|---|---|
| event_id, host_id | Идентификатор окна и хоста |
| timestamp, window_end | Начало и конец минутного окна; тревога доступна в window_end |
| dataset_part | Часть эксперимента |
| connections_per_min | Число соединений за минуту |
| unique_destinations | Число адресов назначения за минуту |
| outbound_mb | Исходящий трафик, Мбайт за минуту |
| score_knn_normal, score_kde_normal | Оценки kNN и KDE, обученных на подтверждённой норме |
| score_knn_mixed, score_kde_mixed | Оценки тех же методов, обученных на смеси без меток |
| score_knn_stability | Оценки kNN на 80 % нормальной обучающей истории для проверки устойчивости |
| is_attack, attack_type, incident_id | Проверочная истина; у нормы incident_id пуст |
| normal_kind | Исходный легитимный фон routine или backup; не является входом детектора |

**Происхождение оценок.** При подготовке использована отдельная история трёх хостов за 01.02.2026 (4320 окон), которая не входит в CSV. В нормальном варианте все окна подтверждённо легитимны. Смешанная история получена из того же фона с добавлением атакоподобного поведения в часть окон; при обучении метки не использованы. Для каждого режима отдельно выполнены `log1p` трёх признаков и `StandardScaler`, настроенный только по его обучающей истории. kNN использует расстояние до 15-го соседа; KDE — отрицательную логарифмическую плотность с гауссовским ядром и шириной 0,15. Вспомогательный kNN обучен на случайных 80 % нормальной истории без возвращения, со своим scaler; выбор поднабора воспроизводим при seed=77. Эти вычисления выполнены при подготовке данных, повторять их на занятии не требуется.

Во всех пяти столбцах score большее значение означает большую подозрительность. Шкалы моделей не сопоставляются как качество.

**Разделение доступа к истине.** Для удобства выдаётся один CSV: истина технически присутствует в нём. До задания 4 используйте только рабочее представление monitoring без `is_attack`, `attack_type`, `incident_id` и `normal_kind`. Это учебное ограничение использования полей, а не сокрытие разметки. В задании 4 подключите заранее сохранённую таблицу истины по `event_id`.

**Заранее заданные условия:** α=0,01; тревога при строгом `score > T`; своевременность — не позднее 3 минут; заявки — по соседним тревогам хоста с промежутком не больше 10 минут; топ-40 — на весь двухсуточный пакет, не на сутки. При равных score используйте стабильную сортировку с сохранением исходного порядка строк.


In [ ]:
# Импорты и заранее заданные условия эксперимента
# Напишите собственную реализацию.


## Задание 1. Подготовка данных и условий проверки

Время — 8 минут. Максимум — 2 балла.

1. Загрузить CSV, преобразовать временные столбцы, проверить уникальность event_id и пары host_id–timestamp, отсутствие повторов и длительность окон
2. Выделить calibration, normal_audit, monitoring и shifted_normal; проверить размеры, периоды и отсутствие общих event_id
3. Создать рабочую таблицу monitoring без is_attack, attack_type, incident_id и normal_kind, а проверочную истину сохранить отдельно до задания 4
4. Зафиксировать α=0,01, строгий порог score > T, срок 3 минуты, интервал объединения 10 минут и бюджет 40 окон на весь двухсуточный пакет

**Результат:** Таблица состава четырёх частей, проверки идентификаторов и окон, отдельные представления мониторинга и истины.


In [ ]:
# Аудит, разделение частей и рабочая таблица без истины
# Напишите собственную реализацию.


**Ваш вывод:** объясните рассчитанные таблицы, их смысл для SOC и ограничения.

…


## Задание 2. Пороги и проверка на нормальных данных

Время — 12 минут. Максимум — 4 балла.

1. Для каждой из четырёх конфигураций определить 99-й процентиль score на calibration и зафиксировать пороги
2. Без изменения порогов рассчитать число и долю тревог на normal_audit; для KDE на норме построить точный двухсторонний 95-процентный биномиальный интервал
3. Для KDE на норме сравнить срабатывания отдельно на routine и backup, затем на normal_audit и shifted_normal
4. Для гипотетического нуля тревог при n=100, 1000 и 4320 вычислить верхнюю одностороннюю 95-процентную границу; объяснить ограничения биномиальных предположений для связанных окон

**Результат:** Пороги, таблица частот, доверительный интервал, разбивка по легитимным режимам и проверка дрейфа.


In [ ]:
# Пороги, независимая норма, интервал, режимы, дрейф и случай нуля тревог
# Напишите собственную реализацию.


**Ваш вывод:** объясните рассчитанные таблицы, их смысл для SOC и ограничения.

…


## Задание 3. Мониторинг без разметки и устойчивость

Время — 15 минут. Максимум — 4 балла.

1. Использовать только рабочее представление monitoring; по фиксированным порогам рассчитать число и долю тревог четырёх моделей
2. Рассчитать длительность наблюдения по началу первого и концу последнего окна; определить суммарные тревоги трёх хостов в сутки и дневные количества
3. Построить график числа тревог по календарным дням средствами matplotlib
4. Для score_knn_normal и score_knn_stability на одних и тех же event_id вычислить корреляцию Спирмена и индекс Жаккара множеств топ-40
5. Объяснить, какие выводы об обнаружении атак невозможны без разметки и почему устойчивость не доказывает правильность

**Результат:** Таблица нагрузки, дневной график, показатели устойчивости и ограничения выводов без истины.


In [ ]:
# Срабатывания и суммарная нагрузка по дням
# Напишите собственную реализацию.


In [ ]:
# График дневной нагрузки
# Напишите собственную реализацию.


In [ ]:
# Спирмен и Жаккар топ-40 по одним и тем же event_id
# Напишите собственную реализацию.


**Ваш вывод:** объясните рассчитанные таблицы и графики, их смысл для SOC и ограничения.

…


## Задание 4. Проверка по подтверждённой истине

Время — 15 минут. Максимум — 4 балла.

1. Подключить проверочную истину к ранее оценённым окнам по event_id с проверкой one-to-one и сохранением соответствия оценок объектам
2. Для четырёх конфигураций определить число атакующих окон с тревогой и без неё, легитимных окон с тревогой и общее число тревог
3. Проверить равенство общего числа тревог сумме атакующих и легитимных срабатываний; сформировать таблицу обнаружения по scan, brute_force и exfiltration
4. Рассчитать число тревог на легитимных окнах в сутки и построить столбчатое сравнение обнаруженных окон по сценариям

**Результат:** Проверенное объединение истины, таблицы по окнам и сценариям, график и оценка ложной нагрузки.


In [ ]:
# Подключение истины, окна и сценарии
# Напишите собственную реализацию.


In [ ]:
# Сравнение обнаружения окон по сценариям
# Напишите собственную реализацию.


**Ваш вывод:** объясните рассчитанные таблицы и графики, их смысл для SOC и ограничения.

…


## Задание 5. Инциденты и практическая полезность для SOC

Время — 20 минут. Максимум — 6 балла.

1. Из всех атакующих окон сформировать каталог истинных инцидентов; считать инцидент обнаруженным при наличии хотя бы одной тревоги на его окнах
2. Для каждой конфигурации вычислить задержку от начала инцидента до первого window_end с тревогой; пропуски записать как NaN
3. Определить число обнаруженных и своевременно обнаруженных инцидентов с задержкой ≤ 3 минут; медиану считать только по обнаруженным, а долю своевременных — по всем известным инцидентам
4. Для KDE на норме объединить соседние тревоги одного хоста в заявки при промежутке ≤ 10 минут, не используя incident_id; сохранить связь заявок с исходными event_id
5. Для каждой конфигурации выбрать топ-40 по убыванию score и определить число и долю атакующих окон и число разных инцидентов в списке
6. Составить final_report и записку SOC из 5–7 предложений, сопоставив обнаружение, задержки, ложную нагрузку, заявки, бюджет и ограничения

**Результат:** Каталог и задержки инцидентов, заявки, результаты топ-40, итоговая таблица и обоснованный вывод SOC.


In [ ]:
# Все инциденты, задержки, обнаружение и своевременность
# Напишите собственную реализацию.


In [ ]:
# Заявки по хосту и времени с сохранением исходных событий
# Напишите собственную реализацию.


In [ ]:
# Топ-40 и итоговый отчёт
# Напишите собственную реализацию.


**Ваш вывод:** объясните рассчитанные таблицы, их смысл для SOC и ограничения.

…


## Записка SOC

В 5–7 предложениях укажите условия эксперимента, реальные числовые результаты, пропуски, своевременность, ложную нагрузку и полезность бюджета. Обоснуйте сравнение конфигураций; отдельно укажите ограничения переноса выводов на реальную инфраструктуру.

**Текст:** …


## Перед сдачей

Сохраните выполненную тетрадь с собственным кодом, таблицами, графиками и пояснениями. Не изменяйте пороги по тестовым результатам. Не приравнивайте минутное окно, заявку и подтверждённый инцидент.
